# Building Custom MCP Tools

Learn how to create custom Model Context Protocol (MCP) tools for the ADEPT framework.

## Network Access Note (Opt-In)

By default, JupyterLab is isolated on `frontend_net` and cannot reach MCP servers directly.
Tool registration demonstrated here uses the Management API via `nginx_proxy`.

If you need direct MCP server access for debugging or local development, add JupyterLab
to `application_net` via docker-compose override:

```yaml
# deployment/local/docker-compose/docker-compose.override.yaml
services:
  jupyterlab:
    networks:
      - adept_frontend_network
      - adept_application_network  # Opt-in: direct MCP access
```

Then rebuild: `make rebuild-jupyterlab`

## Topics Covered
- MCP tool architecture
- Creating tool functions
- Tool registration via Management API
- Testing and debugging
- Deployment

## Docker Configuration
- JupyterLab service: `deployment/local/docker-compose/docker-compose.core.yaml`
- Rebuild after config changes: `make rebuild-jupyterlab`

In [ ]:
# Example MCP tool structure
from fastmcp import FastMCP
from datetime import datetime

# Initialize MCP server
mcp = FastMCP("Custom Tools")

@mcp.tool()
async def custom_analysis(data: str, method: str = "statistical"):
    '''Perform custom analysis on data.
    
    Args:
        data: Input data to analyze
        method: Analysis method (statistical, ml, etc.)
    '''
    # Tool implementation
    result = f"Analyzed {len(data)} characters using {method} method"
    return {"result": result, "timestamp": str(datetime.now())}

print("✅ Custom MCP tool defined")

## Registering Tools

In [ ]:
# Tool registration example
# Tools are registered via the Management API through nginx_proxy.
# The endpoint field specifies where the orchestration service will reach
# the MCP server (on application_net — NOT accessible from JupyterLab by default).
import json
import os

config = json.load(open("/tmp/adept_config.json"))
API_BASE_URL = config.get('api_base_url', os.getenv('ADEPT_API_BASE_URL', 'https://nginx_proxy/v1'))

tool_config = {
    "name": "custom_analysis",
    "description": "Custom data analysis tool",
    "parameters": {
        "data": {"type": "string", "required": True},
        "method": {"type": "string", "default": "statistical"}
    },
    "endpoint": "http://custom_mcp_server:8090"  # Internal address (application_net)
}

print("Tool registration config (submitted via Management API):")
print(f"Registration endpoint: {API_BASE_URL}/admin/tools")
print(f"Tool endpoint (internal): {tool_config['endpoint']}")
print()
print(json.dumps(tool_config, indent=2))